# Memory Estimates Playground — coding workshop (beginner)

**The Colab version of the lesson exercise "Which models fit on a 16 GB GPU?"**
Expected time: ~8–10 minutes, individual, no GPU needed.

The lesson asks you to estimate the weight memory for 7B, 32B, and 70B-parameter models in
FP16, INT8, and INT4, using `memory = parameters * bits / 8`, and to check whether each
result would fit into 16 GB of GPU memory. Here you do that same calculation in code
instead of on paper, and check it against a model size the lesson already worked out for
you before moving on to the bigger ones.

**What you will learn:**

- to compute the memory a model's weights take from its number of parameters and bits: parameters × bits ÷ 8
  bytes;
- to check that formula against the lesson's numbers for our 1.5B model;
- to decide which precisions of a 7B, a 32B and a 70B model fit on a 16 GB GPU, and the most precise one that
  fits;
- when quantization stops being a saving and becomes the only way to fit a model, and when even 4 bits are
  not enough.


## How this workshop works

This is the coding version of the notebook, **beginner level**. Some code cells have gaps
marked `___`. An exercise above each such cell says what to write. This version takes
longer than the expected time given above.

1. Read the exercise. Open the hints if you need them.
2. Replace every `___` in the cell with your code, then run the cell.
3. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
4. If you are stuck, the solution is below the check cell. Open it after you have tried.
   Or tick **fill_in_solutions** in the cell right below these instructions and run that cell:
   from then on, every exercise cell you run gets its solution in place of its code. With the box
   ticked, Runtime → Run all runs the whole notebook with the solutions.

A cell that still contains `___` stops with `ExerciseGaps: this cell still has gaps marked ___`.
Later cells use your code, so finish each exercise before moving on.

In [ ]:
# @title Run this cell first. Tick the box to fill in the solutions (point 4 above).
fill_in_solutions = False  # @param {type:"boolean"}

import io
import re
import tokenize

_solutions = {
    1: "def weight_memory_gb(params_billion, bits):\n    params = params_billion * 1e9\n    bytes_total = params * bits / 8\n    return bytes_total / 1e9  # 1 GB = 10**9 bytes\n\n\nfor bits, label in [(32, \"FP32\"), (16, \"FP16/BF16\"), (8, \"INT8\"), (4, \"INT4\")]:\n    gb = weight_memory_gb(1.5, bits)\n    print(f\"1.5B model, {label}: {gb:.2f} GB  (lesson: ~{ {32:6, 16:3, 8:1.5, 4:0.75}[bits] } GB)\")",
    2: "GPU_MEMORY_GB = 16  # a Colab T4\n\nfor model_size in [7, 32, 70]:\n    for bits, label in [(16, \"FP16\"), (8, \"INT8\"), (4, \"INT4\")]:\n        gb = weight_memory_gb(model_size, bits)\n        fits = \"fits\" if gb < GPU_MEMORY_GB else \"does NOT fit\"\n        print(f\"{model_size}B model, {label}: {gb:.2f} GB --- {fits} in {GPU_MEMORY_GB} GB\")\n    print()\n\n\ndef most_precise_fitting_bits(model_size, gpu_gb=GPU_MEMORY_GB, options=(16, 8, 4, 3, 2)):\n    for bits in options:\n        if weight_memory_gb(model_size, bits) < gpu_gb:\n            return bits\n    return None\n\n\nfor model_size in [7, 32, 70]:\n    print(f\"{model_size}B: highest precision that fits = {most_precise_fitting_bits(model_size)} bits\")",
}
_fill_in = set(_solutions) if fill_in_solutions else set()


class ExerciseGaps(Exception):
    """A cell still has gaps marked ___."""


def _has_gap(lines):
    """True if `___` appears in the cell as code (not in a string or a comment)."""
    code = "".join(l for l in lines if not l.lstrip().startswith(("%", "!")))
    try:
        return any(t.type == tokenize.NAME and t.string == "___"
                   for t in tokenize.generate_tokens(io.StringIO(code).readline))
    except (tokenize.TokenError, SyntaxError):
        return False


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution.
    A cell that still has gaps ___ stops with a message. (In IPython, ___ is a built-in name, the
    third-to-last output, so without this check the cell would run on with a wrong value.)"""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if m and int(m.group(1)) in _fill_in:
        n = int(m.group(1))
        return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)
    if _has_gap(lines):
        return ['raise ExerciseGaps("this cell still has gaps marked ___: fill them in and run it again, '
                'or tick fill_in_solutions in the first code cell and run that cell")\n']
    return lines


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)
if fill_in_solutions:
    print("✓ Every exercise cell you run from now on gets its solution.")

## Step 1 — the formula, checked against a number you already have

The lesson screen "Memory for the weights of our model" already gave you the answer for
Qwen2.5-1.5B: about 6 GB at FP32, 3 GB at FP16/BF16, 1.5 GB at INT8, 0.75 GB at INT4. Run
the same formula here for that same model size, as a sanity check that the formula and
the lesson agree before you use it on a new number.


### ✏️ Exercise 1 — Weight memory from parameter count and bits

**Goal:** turn a number of parameters and a number of bits into GB of weights, the formula of the lesson's memory screen.

Write `weight_memory_gb`: the memory for the weights alone, in GB (1 GB = 10⁹
bytes, as on the lesson's memory screen), for a model with `params_billion` billion parameters stored at `bits` bits
each.

<details><summary><b>Hints</b> (click to open)</summary>

- One billion is `1e9`.
- There are 8 bits in a byte.
- 1 GB here is 10⁹ bytes, so our 1.5B model in 16 bits comes out at exactly 3 GB.

</details>

In [ ]:
# Exercise 1
def weight_memory_gb(params_billion, bits):
    params = params_billion * ___
    bytes_total = params * bits / ___
    return bytes_total / ___


for bits, label in [(32, "FP32"), (16, "FP16/BF16"), (8, "INT8"), (4, "INT4")]:
    gb = weight_memory_gb(1.5, bits)
    print(f"1.5B model, {label}: {gb:.2f} GB  (lesson: ~{ {32:6, 16:3, 8:1.5, 4:0.75}[bits] } GB)")


In [ ]:
# Check for Exercise 1. Run this cell as is.
assert abs(weight_memory_gb(1.5, 16) - 3.0) < 1e-9, weight_memory_gb(1.5, 16)
assert abs(weight_memory_gb(7, 4) - 3.5) < 1e-9, weight_memory_gb(7, 4)
assert abs(weight_memory_gb(1.5, 32) / weight_memory_gb(1.5, 8) - 4) < 1e-9
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def weight_memory_gb(params_billion, bits):
    params = params_billion * 1e9
    bytes_total = params * bits / 8
    return bytes_total / 1e9  # 1 GB = 10**9 bytes


for bits, label in [(32, "FP32"), (16, "FP16/BF16"), (8, "INT8"), (4, "INT4")]:
    gb = weight_memory_gb(1.5, bits)
    print(f"1.5B model, {label}: {gb:.2f} GB  (lesson: ~{ {32:6, 16:3, 8:1.5, 4:0.75}[bits] } GB)")
```

</details>

## Step 2 — now the actual activity: 7B, 32B, and 70B on a 16 GB GPU

A Colab T4 has 16 GB of GPU memory. `GPU_MEMORY_GB` below is deliberately just that one
number, ignoring the activations (intermediate values computed for each input), the KV cache, and the memory the software itself takes, exactly as the lesson
says to for this first pass. 7B fits, but in FP16 it leaves little room for anything else; 32B and 70B are
where the question stops being "does quantization help" and starts being "does the model
fit at all."


### ✏️ Exercise 2 — Does it fit, and at what precision?

**Goal:** find, for each model size, the most precise format (the most bits) whose weights still fit in 16 GB.

Two parts:

1. Decide whether each configuration fits in `GPU_MEMORY_GB`.
2. Write `most_precise_fitting_bits(model_size)`: go through the precisions in `options`, from
   most to fewest bits, and return the first one whose weights fit. Return `None` if none
   fits.

<details><summary><b>Hints</b> (click to open)</summary>

- Use `weight_memory_gb(model_size, bits)` from Step 1.
- Return inside the loop as soon as one fits; the loop goes from most to fewest bits.

</details>

In [ ]:
# Exercise 2
GPU_MEMORY_GB = 16  # a Colab T4

for model_size in [7, 32, 70]:
    for bits, label in [(16, "FP16"), (8, "INT8"), (4, "INT4")]:
        gb = weight_memory_gb(model_size, bits)
        fits = "fits" if ___ else "does NOT fit"
        print(f"{model_size}B model, {label}: {gb:.2f} GB --- {fits} in {GPU_MEMORY_GB} GB")
    print()


def most_precise_fitting_bits(model_size, gpu_gb=GPU_MEMORY_GB, options=(16, 8, 4, 3, 2)):
    for bits in options:
        if ___:
            return ___
    return None


for model_size in [7, 32, 70]:
    print(f"{model_size}B: highest precision that fits = {most_precise_fitting_bits(model_size)} bits")


In [ ]:
# Check for Exercise 2. Run this cell as is.
assert [most_precise_fitting_bits(s) for s in (7, 32, 70)] == [16, 3, None], \
    [most_precise_fitting_bits(s) for s in (7, 32, 70)]
assert most_precise_fitting_bits(70, gpu_gb=80) == 8
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
GPU_MEMORY_GB = 16  # a Colab T4

for model_size in [7, 32, 70]:
    for bits, label in [(16, "FP16"), (8, "INT8"), (4, "INT4")]:
        gb = weight_memory_gb(model_size, bits)
        fits = "fits" if gb < GPU_MEMORY_GB else "does NOT fit"
        print(f"{model_size}B model, {label}: {gb:.2f} GB --- {fits} in {GPU_MEMORY_GB} GB")
    print()


def most_precise_fitting_bits(model_size, gpu_gb=GPU_MEMORY_GB, options=(16, 8, 4, 3, 2)):
    for bits in options:
        if weight_memory_gb(model_size, bits) < gpu_gb:
            return bits
    return None


for model_size in [7, 32, 70]:
    print(f"{model_size}B: highest precision that fits = {most_precise_fitting_bits(model_size)} bits")
```

</details>

**Reflection:** Look at your printed results for 32B and 70B. At 32B, which precisions fit
and which don't? At 70B? What does this tell you about when quantization changes from
"helps efficiency" to "the only way to fit the model at all" --- and is there a point where
quantization alone isn't enough either?


<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

**32B:** FP16 (64 GB) and INT8 (32 GB) are far too big. INT4 comes out at exactly 16 GB, which the check counts as not fitting: it would fill the whole GPU and leave nothing for anything else. `most_precise_fitting_bits` finds that 3 bits (12 GB) is the highest precision that fits.

**70B:** none of the printed precisions fits, and even 2 bits needs 17.5 GB, so `most_precise_fitting_bits` returns `None`.

For a 7B model, quantization helps efficiency: FP16 fits on paper (14 GB of 16; with Qwen2.5-7B's 7.6 billion parameters, 15.2 GB, under 1 GB to spare), and fewer bits only free memory. Notebook 02c's Part C tries the fp16 load: in our run part of the model had to move to CPU memory, and on a T4 expect the same or a failed load. At 32B, quantization is the only way to load the model at all on this GPU, and only at 3 bits, where Notebook 02b showed the rounding error growing fast. At 70B, quantization alone is not enough: you need a bigger GPU (with 80 GB, 8 bits fits), several GPUs, or a smaller model. And these are best cases: the KV cache and the model's working memory need room too, so the real limits come sooner.

</details>

## What you learned: check yourself

Answer each question from what you ran and read above. If one is hard, go back to the
step it comes from.

1. How many GB do the weights of our 1.5B model take at 16 bits? At 4 bits?
2. Which precisions of the 7B model fit in 16 GB? Which of the 32B model?
3. Why does the check count a model whose weights need exactly 16 GB as not fitting?
4. What does this estimate leave out, so that the real memory a model needs is higher?

## Code literacy check

Before you move on, add `#` comments to the exercise cells above explaining, in your own
words, what each line does and why it's there, in language a classmate who hasn't seen
the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?
